In [ ]:
#@title 按這裡開始（先按 ▶）
print('✅ W03 出發！本週目標：把全班表單資料讀進 Colab，自己寫查詢與缺失值處理')
DEMO = ('https://raw.githubusercontent.com/myliao2007/stust-course-1151/'
        'main/ai-intro-pc/data/demo_class.csv')  # ←投影片未含，執行所需：班級資料備援
print('老師還沒公布試算表 ID 也沒關係，下一格留空白就會自動讀示範資料。')

# W03　自己寫程式整理全班資料

**先存副本**：`檔案 → 在雲端硬碟中儲存副本`，檔名 `AI導論_W03_學號_姓名`。

## 任務一：從填表單到讀進 Colab

1. **先填表單**：三題 —— 身高公分、單程通勤分鐘、每天手機時數
2. **老師發布試算表**：共用 → 改成「知道連結的人可以檢視」
3. **複製試算表 ID**：網址 `/d/` 與 `/edit` 中間那一長串英數字
4. **貼進下一格再執行**：只改 `SHEET_ID` 那一行，其餘不要動
5. **確認 shape 對不對**：欄數不是 4，代表表單題目被改過

> 課後補做、或老師還沒公布 ID 時，`SHEET_ID` 留空白即可，
> 程式會自動改讀課程附的示範資料（`ai-intro-pc/data/demo_class.csv`）。

## 任務二之一：讀進資料並改成英文欄名

**只改第 1 行的試算表 ID，其餘完全不要動。**

**會看到**：`shape` 印出（人數, 4）；讀不到多半是 ID 貼錯或多了空白。
用示範資料時會是 `(35, 3)`，沒有 `ts` 欄。

In [ ]:
SHEET_ID = ''    # 貼上老師給的 ID（網址 /d/ 與 /edit 中間那串）；留空白＝用示範資料

import pandas as pd
BASE = 'https://docs.google.com/spreadsheets/d/'
if SHEET_ID.strip():
    df = pd.read_csv(BASE + SHEET_ID + '/export?format=csv')
    # 中文欄名打字麻煩，這一行統一改成英文
    df.columns = ['ts', 'height_cm', 'commute_min', 'phone_hours']
else:                       # ←投影片未含，執行所需
    df = pd.read_csv(DEMO)  # ←投影片未含，執行所需：沒 ID 就讀課程示範資料
print(df.shape)
df.head()

## 任務二之二：拿到資料先跑這五個指令

這五個是固定動作，**往後每一週拿到新資料都先跑一次**。

**會看到**：形狀、每欄型態與非空筆數、統計摘要、每欄缺幾筆、前五列。

In [ ]:
print('形狀：', df.shape)      # 幾列幾欄
df.info()                      # 每欄型態與非空筆數
display(df.describe())         # 數值欄的統計摘要
print(df.isna().sum())         # 每一欄各缺幾筆
display(df.head())             # 前五筆長什麼樣

# 想一想：info() 說 height_cm 是 object，代表什麼？

**想一想的答案**：型態是 `object` 代表整欄變成文字，通常有人填了「約 30 分」。

### 這份班級資料長什麼樣

| 英文欄名 | 表單題目 | 單位與合理範圍 |
|---|---|---|
| `height_cm` | 你的身高 | 公分，約 145 到 200 |
| `commute_min` | 單程通勤時間 | 分鐘，0 到 120 |
| `phone_hours` | 每天用手機幾小時 | 小時，0 到 16 |
| `ts` | 表單自動記錄的時間 | 分析用不到，可以刪掉 |
| 缺失值 | 有人跳過沒填 | 顯示成 `NaN`，任務四要處理 |
| 離群值 | 身高 1720、通勤 600 | 多半是打錯字，先看再決定 |

---

## 任務三：自己寫 describe、篩選、groupby

三段各留一個 `____`，**寫對了 print 才有東西**。

**會看到**：統計表、通勤超過 30 分的人數，以及兩組各自的平均手機時數。

In [ ]:
# ① 只看三個數值欄的統計摘要
cols = ['height_cm', 'commute_min', 'phone_hours']
display(df[cols].____())         # ← 提示：統計摘要

# ② 篩出通勤超過 30 分鐘的同學
far = df[____]                   # ← 提示：commute_min 大於 30
print('通勤超過 30 分的有', len(far), '人')

# ③ 依「是不是通勤族」分組，比較平均手機時數
df['far'] = df['commute_min'] > 30
print(df.groupby('far')['phone_hours'].____())  # ← 提示：平均

## 任務四之前：三種處理各要記什麼

| 做法 | 要記下來的數字 | 副作用 |
|---|---|---|
| **1 刪除 `dropna`** | 剩幾筆、平均變多少 | 缺很多時會把資料刪光，順便看刪掉幾成 |
| **2 平均填補 `mean`** | 平均與標準差 | 平均不會變，但標準差變小，資料看起來比實際集中 |
| **3 中位數填補 `median`** | 平均與標準差 | 有離群值時比平均穩，是最常用的預設做法 |

## 任務四：缺失值三種處理，各跑一次

把 `method` 換成 `'median'`、`'mean'`、`'drop'` **各跑一次，三組數字抄下來**。

**會看到**：三次的筆數與標準差都不同 —— 這就是報告要寫「怎麼處理」的理由。

> 用課程示範資料時 `height_cm` 只缺 1 筆，所以 `mean` 與 `median` 兩種填補的結果幾乎一樣；
> 全班真實資料缺得多，差距才會明顯。`drop` 那一次的筆數一定會少，先看這一欄。

In [ ]:
method = 'median'    # 換成 'mean'、'drop' 各跑一次
col = 'height_cm'

d = df.copy()
print('處理前缺幾筆：', d[col].isna().sum())

if method == 'drop':
    d = ____                      # ← ① 刪掉這一欄有缺的整列
else:
    v = d[col].mean() if method == 'mean' else d[col].____()
    d[col] = d[col].fillna(v)     # ← ② 上一行的 ____ 填中位數

print(method, '剩', len(d), '筆　平均',
      round(d[col].mean(), 1), '標準差', round(d[col].std(), 2))

## 任務五（進階）：自己寫一個 clean() 函式

把前面做過的四件事，**包成一個可以重複用的函式**。

**會看到**：一行 `clean(df)` 就把去重、轉型、離群、填補四件事全做完，
最後印出整理後的 shape 與平均身高。

In [ ]:
def clean(data):
    d = data.copy()
    d = d.drop_duplicates()               # 一、去掉重複
    d['height_cm'] = pd.to_numeric(
        d['height_cm'], errors='coerce')  # 二、文字轉數值
    d.loc[d['height_cm'] > 300, 'height_cm'] = None
    d = ____                              # ← ① 三、填補缺失值
    return ____                           # ← ② 交回整理好的表

out = clean(df)
print(out.shape, round(out['height_cm'].mean(), 1))

## 繳交：這一週要交什麼

1. **任務三的三段輸出留在筆記本裡**：統計表、通勤超過 30 分的人數、兩組平均手機時數
2. **任務四的三組數字整理成一段文字**：刪除／平均／中位數各剩幾筆、平均與標準差各多少
3. **寫一句話說明你選哪一種，為什麼** ← 這一句是評分重點，只貼數字不寫理由不給分
4. **確認已儲存、改共用權限，再交連結**：寫出 `clean()` 的人在備註欄寫「已完成任務五」

### 常見狀況排除

| 狀況 | 原因 | 怎麼處理 |
|---|---|---|
| 讀回來全是 HTML | 試算表沒開共用權限 | 共用 → 知道連結的人可以檢視 |
| `ParserError` | ID 多複製了空白或斜線 | 只留 `/d/` 與 `/edit` 中間那一段 |
| 欄數不是 4 | 表單題目被改過或多一題 | 重看試算表標題列再對 `columns` |
| 整欄變成 object | 有人填「約 30 分」 | `pd.to_numeric(..., errors='coerce')` |
| `KeyError` 欄名找不到 | 英文欄名打錯或大小寫錯 | `print(df.columns)` 對一次 |
| 改了卻沒有變 | 改到 `df`，看的是 `d` | 確認自己印的是哪一個變數 |
| 筆數每次都不一樣 | 同學還在陸續填表單 | 重跑第 1 格會抓到最新的資料 |
| 下課後檔案不見 | 存在教室電腦桌面 | 一律存回雲端硬碟，不要存本機 |

### 延伸挑戰（A 到 C 越後面越難）

- **A 改參數再跑**：把 `col` 換成 `commute_min` 與 `phone_hours`，各跑一次任務四。三欄的缺失比例一樣嗎？
- **B 換一種做法**：用 `pd.cut()` 把身高切成矮／中／高三段，再 `groupby` 比較平均通勤時間。
- **C 說出為什麼**：為什麼用平均填補之後標準差會變小？拿任務四印出來的兩個標準差，講給組員聽。

---

<details>
<summary><b>參考解</b>（真的卡住再打開，先自己試滿 10 分鐘）</summary>

**任務三**

```python
display(df[cols].describe())
far = df[df['commute_min'] > 30]
print(df.groupby('far')['phone_hours'].mean())
```

**任務四**

```python
if method == 'drop':
    d = d.dropna(subset=[col])                 # ① 只刪這一欄有缺的整列
else:
    v = d[col].mean() if method == 'mean' else d[col].median()   # ②
```

**任務五**

```python
    d = d.fillna(d.median(numeric_only=True))  # ① 用中位數填補所有數值欄
    return d                                   # ② 交回整理好的表
```

</details>